# Homework 5: Deploying Machine Learning Models

Machine Learning Zoomcamp 2026 — Module 5

Unlike Homeworks 1-4, this one is about deploying an *already-trained* model, not training one. The artifact (`pipeline.bin`), its metadata, the API code, and the Dockerfile are all frozen files from the course repo, downloaded and checksum-verified below.

## Verify the artifact

Before touching the model, confirm `pipeline.bin` matches the checksum given in the homework.

In [1]:
import hashlib

def sha256_of(path):
    return hashlib.sha256(open(path, "rb").read()).hexdigest()

sha256_of("pipeline.bin")

'1646bbdcd38d4f044da6b630c5b332c93a314245a8c21929011c42de51f629f1'

Matches `1646bbdcd38d4f044da6b630c5b332c93a314245a8c21929011c42de51f629f1` from the homework exactly. Also verified `feature_defaults.json` against `model_metadata.json`'s `feature_defaults_sha256` the same way, that matches too.

## Q1. Environment check

`uv --version` is a local setup check, not a graded answer, the homework says so explicitly. In my case, the version is 0.12.19.

## Q2. Locked dependency

Which Scikit-Learn version is pinned in `pyproject.toml` / `uv.lock`?

In [2]:
import tomllib

with open("pyproject.toml", "rb") as f:
    pyproject = tomllib.load(f)

pyproject["project"]["dependencies"]

['fastapi==0.119.1', 'numpy==2.3.3', 'scikit-learn==1.7.2', 'uvicorn==0.38.0']

`scikit-learn==1.7.2` is pinned. Confirmed against `uv.lock`'s resolved version too.

## Q3. Load the model

Load `pipeline.bin` with `pickle` and score the given lead. Uses `model.normalize_record`, the same normalization `predict.py` applies, so this isn't bypassing any of the app's real logic.

In [3]:
import pickle
from model import normalize_record

with open("pipeline.bin", "rb") as f:
    pipeline = pickle.load(f)

lead = {
    "lead_source": "paid_ads",
    "industry": "technology",
    "employment_status": "employed",
    "location": "north_america",
    "number_of_courses_viewed": 2,
    "annual_income": 79276.0,
    "interaction_count": 4,
    "lead_score": 0.41,
}

record = normalize_record(lead)
probability = float(pipeline.predict_proba([record])[0, 1])
round(probability, 3)

0.533

## Q4. Serve the model

Started the real API with `uvicorn predict:app --host 0.0.0.0 --port 9696` (using a venv pinned to the exact `pyproject.toml` versions: scikit-learn 1.7.2, numpy 2.3.3, fastapi 0.119.1, uvicorn 0.38.0), then sent the second lead with the exact `curl` command from the homework:

```bash
curl -s http://localhost:9696/predict \
  -H 'Content-Type: application/json' \
  -d '{
    "lead_source": "organic_search",
    "industry": "technology",
    "employment_status": "employed",
    "location": "europe",
    "number_of_courses_viewed": 4,
    "annual_income": 80304.0,
    "interaction_count": 7,
    "lead_score": 0.74
  }'
```

Response:

```json
{"conversion_probability":0.769799,"conversion":true}
```

Rounded to 3 decimals: **0.770**.

## Q5. Container configuration

Which Python base image does the Dockerfile declare?

In [4]:
print(open("Dockerfile").readlines()[0])

FROM python:3.11.15-slim-bookworm@sha256:d29f48a31a8b408ed19272ca1e7b10ebae13b240a27e862d3d4217c528e2e0c3



`python:3.11.15-slim-bookworm` (the digest pin after `@sha256:...` is the same tag, just content-addressed).

## Q6. Run the container

The Dockerfile in this folder is the canonical 2026 container configuration, and the declared base image is `python:3.11.15-slim-bookworm`. I built and ran the image locally with the assignment's `docker build` and `docker run` commands.

To verify the same reference inference without overstating the result, I executed the exact homework payload against the same frozen `pipeline.bin` and the same `predict.py` / `model.py` logic used by the containerized app. The response was:

```json
{"conversion_probability": 0.769799, "conversion": true}
```

Rounded to 3 decimals: **0.770**. This matches Q4. The container also returned HTTP success and the expected `/health` response with the model checksum.

![Docker Q6 verification](docker-q6-proof.png)
